# OCR Pipeline Specification for KCAC Kurdish Dataset v0.1

> **For**: AI coding agent (Codex / Claude Code / Cursor)
> **Goal**: Build a Python pipeline that converts ~400 page scans from KCAC
> into a research-grade OCR ground-truth dataset in 30 days, using multi-engine
> bootstrap and human-in-the-loop correction.
> **Author**: Zebari Hiwa Fakhr Abdulkhaleq, PhD candidate, TUSUR University
> **Output format target**: PAGE XML (PRImA standard) + HuggingFace `datasets` derivative

---

## 1. Project Overview

This pipeline takes scanned page images from a Kurdish digital library (KCAC),
produces multiple OCR hypotheses for each line, cross-validates them, exports
the result to **PAGE XML** for human correction in **eScriptorium**, and
finally produces a research-grade ground-truth dataset for OCR model training
and benchmarking on historical Central Kurdish (Sorani) print.

The pipeline is designed for **30-day delivery of v0.1** with 5–10 books and
300–500 fully annotated pages.

The library content is publicly accessible and the user has direct
collaboration with the institution for scan provision. Inputs are local image
files; the pipeline performs no web scraping.

---

## 2. Inputs and Outputs

### Inputs (provided locally)
```
input/
├── books.jsonl              # bibliographic catalog from KCAC
├── images/
│   ├── kcac_000152/
│   │   ├── page_0001.jpg    # original-resolution scans, 300+ DPI
│   │   ├── page_0002.jpg
│   │   └── ...
│   └── ...
└── config.yaml              # pipeline configuration
```

### Outputs
```
output/
├── ocr_raw/
│   └── {book_id}/{page_id}/
│       ├── tesseract.json           # per-line text + bbox + confidence
│       ├── kraken.json
│       ├── calamari.json
│       ├── qwen2vl.json
│       └── claude.json              # one file per engine
├── consensus/
│   └── {book_id}/{page_id}.json     # cross-validated, confidence-scored
├── page_xml/
│   └── {book_id}/{page_id}.xml      # PAGE XML for eScriptorium import
├── splits/
│   ├── train_books.txt
│   ├── val_books.txt
│   └── test_books.txt
├── benchmark/
│   ├── baseline_results.json        # Tesseract / Kraken / Calamari CER per bucket
│   └── confusion_examples/
└── reports/
    ├── line_confidence_histogram.png
    ├── disagreement_matrix.csv
    └── coverage_matrix.csv
```

---

## 3. Components to Build

### Component 1: Multi-Engine OCR Bootstrap

**File**: `pipeline/bootstrap.py`

For every page image, produce OCR output from FIVE engines:

| Engine | Library | Purpose |
|--------|---------|---------|
| Tesseract `ckb` | `pytesseract` | Sorani baseline; line-level text + bbox |
| Kraken | `kraken` | Arabic-script historical print (best baselines exist) |
| Calamari | `calamari-ocr` | Letterpress / historical fonts |
| Qwen2-VL-72B | HuggingFace `transformers` | Vision-language fallback (high quality) |
| Claude 3.7 Sonnet | Anthropic API | Vision-language fallback (highest quality on hard pages) |

For each engine, store output as JSON:
```json
{
  "page_id": "kcac_000152_p0007",
  "engine": "qwen2vl",
  "engine_version": "Qwen/Qwen2-VL-72B-Instruct",
  "lines": [
    {
      "line_id": "kcac_000152_p0007_l0001",
      "polygon": [[x1,y1], [x2,y2], ...],
      "baseline": [[x1,y1], [x2,y2]],
      "text": "تروسکایییەک لە ژیانی حەسەن زیرەک",
      "confidence": 0.87
    }
  ]
}
```

**Important behaviour**:
- All engines must produce line polygons in the SAME coordinate system (page pixels).
- For VLMs (Qwen2-VL, Claude), if they cannot produce bounding boxes natively,
  use a separate line-detection model (Kraken) to provide polygons, then ask
  the VLM to transcribe each line crop. Do NOT ask the VLM to do layout +
  transcription jointly — quality drops.
- Run engines sequentially (not in parallel) to keep API costs predictable
  and respect rate limits.

### Component 2: Cross-Validation and Consensus

**File**: `pipeline/consensus.py`

For each page:
1. Align line outputs across engines using polygon IoU (≥ 0.5 threshold).
2. For each aligned line, collect the 5 transcriptions.
3. Compute pairwise character-level edit distance.
4. Assign a confidence label:
   - `auto_accept` — 3+ engines produce identical text (after Unicode
     normalisation), no human review needed
   - `near_agreement` — 3+ engines within 2 character edits of consensus,
     light human review
   - `disagreement` — engines diverge significantly, full human correction
5. Pick the consensus text using majority vote; if tied, prefer the engine
   with the highest historical accuracy on similar pages (configurable).

**Output per page**:
```json
{
  "page_id": "kcac_000152_p0007",
  "lines": [
    {
      "line_id": "kcac_000152_p0007_l0001",
      "polygon": [...],
      "baseline": [...],
      "text_consensus": "تروسکایییەک لە ژیانی حەسەن زیرەک",
      "confidence_label": "auto_accept",
      "engine_outputs": {
        "tesseract": "...",
        "kraken": "...",
        "calamari": "...",
        "qwen2vl": "...",
        "claude": "..."
      },
      "max_pairwise_distance": 0
    }
  ]
}
```

### Component 3: Unicode Normalisation Layer

**File**: `pipeline/normalise.py`

Implement a documented, deterministic Sorani Unicode normaliser. Critical
mappings:

| From | To | Reason |
|------|-----|--------|
| `ي` (U+064A, Arabic Yeh) | `ی` (U+06CC, Persian/Kurdish Yeh) | Modern Sorani standard |
| `ك` (U+0643, Arabic Kaf) | `ک` (U+06A9, Persian/Kurdish Kaf) | Modern Sorani standard |
| Optional tashkeel marks | strip | Sorani convention |
| Multiple combining mark order | NFC | Unicode normalisation |

**Critical rule**: NEVER apply normalisation to the raw layer. Always store
both `text_raw` (as-printed) and `text_normalised` (post-normaliser). The
PAGE XML output must include both as separate `<TextEquiv index="N">` elements.

Provide unit tests with explicit before/after pairs for every documented
mapping. The normaliser must be reversible-traceable (a log of every change
made per line).

### Component 4: PAGE XML Export

**File**: `pipeline/pagexml_export.py`

For every page, produce a valid PAGE XML 2019-07-15 file with:
- Page-level metadata (image filename, dimensions, reading direction)
- One `<TextRegion>` per logical region (for v0.1, treat the whole text body
  as a single region; layout analysis is deferred to v0.5)
- One `<TextLine>` per detected line, containing:
  - `<Coords points="..."/>` (polygon)
  - `<Baseline points="..."/>`
  - `<TextEquiv index="1">` with raw text (as-printed)
  - `<TextEquiv index="2">` with normalised text
  - Custom attributes for `confidence_label`, `engine_consensus_count`

The output must validate against the official PAGE XML schema (XSD), which
is published by PRImA Research Lab.

### Component 5: eScriptorium Import Manifest

**File**: `pipeline/escriptorium_import.py`

Generate a directory layout that eScriptorium can import directly:
- One subfolder per book
- For each page: image file + PAGE XML side by side
- A `manifest.json` with import metadata

This enables the human annotators to start correcting on Day 4 without any
manual setup.

### Component 6: Active Learning Queue

**File**: `pipeline/queue.py`

Produce a prioritised queue of pages for human review, sorted by:
1. **Highest disagreement** first (most learning signal)
2. **Diversity matrix coverage** — early pages should fill diversity buckets
3. **Lines per page** — denser pages first (more data per minute of work)

Output: `output/annotation_queue.csv` with columns:
`page_id`, `book_id`, `priority_score`, `disagreement_lines`, `total_lines`,
`era`, `script`, `typography`, `assigned_to`, `status`.

### Component 7: Quality Reports and Dashboards

**File**: `pipeline/reports.py`

Produce three reports after each pipeline run:

1. **`line_confidence_histogram.png`** — distribution of consensus confidence
   across all lines
2. **`disagreement_matrix.csv`** — pairwise engine agreement rates per book
3. **`coverage_matrix.csv`** — diversity-matrix cells populated so far

Also produce a daily Markdown summary (`reports/daily_{date}.md`) for the
project lead.

### Component 8: Baseline OCR Benchmarking

**File**: `pipeline/benchmark.py`

After human correction is complete on the test split, evaluate each engine
against the gold ground truth:

- **Character Error Rate (CER)** per engine, per script bucket, per era,
  per typography bucket
- **Word Error Rate (WER)** same buckets
- **Line-level accuracy** (% of lines with zero edits)

Produce `benchmark/baseline_results.json` and a Markdown report.

---

## 4. Hard Requirements

### Politeness and cost
- Configurable rate limits per engine (especially Claude / Qwen2-VL APIs)
- Budget tracking: total API spend reported per book, per day, in USD
- Local engines (Tesseract, Kraken, Calamari) run on CPU or GPU — make GPU
  optional, not required
- Resumable: if the pipeline is killed, rerun must skip already-completed
  pages (per-engine, per-page granularity)

### Reliability
- Retry network errors up to 3 times with exponential backoff
- After 5 consecutive API failures, exit gracefully (don't burn budget)
- Every page processed must produce SOME output, even on partial engine
  failure — record which engines failed in the per-page JSON

### Reproducibility
- Pin all model versions (engines, VLMs) in `requirements.txt`
- Single `config.yaml` controls all parameters; commit example
- Include a `Dockerfile` so the pipeline runs identically anywhere
- Log every pipeline action with timestamps; logs survive resumes

### Code quality
- Python 3.10+ with type hints throughout (`from __future__ import annotations`)
- Modular: every component is independently runnable for debugging
- Unit tests with `pytest` for: Unicode normalisation, polygon IoU alignment,
  consensus voting, PAGE XML schema validation
- `ruff` clean, `mypy --strict` clean on the core modules
- Single-process by default; concurrency is opt-in via flag

---

## 5. Configuration File Example

```yaml
# config.yaml

input:
  books_jsonl: input/books.jsonl
  images_root: input/images/

output:
  root: output/

engines:
  tesseract:
    enabled: true
    lang: ckb
    psm: 6
  kraken:
    enabled: true
    model: kraken_arabic_historical_v1
  calamari:
    enabled: true
    model: calamari_arabic_v1
  qwen2vl:
    enabled: true
    model_id: Qwen/Qwen2-VL-72B-Instruct
    use_local_gpu: true
    rate_limit_rpm: 30
  claude:
    enabled: true
    model: claude-sonnet-4-7-20260101
    rate_limit_rpm: 50
    monthly_budget_usd: 200

consensus:
  iou_threshold_for_line_alignment: 0.5
  auto_accept_min_agreeing_engines: 3
  auto_accept_max_pairwise_edits: 0

normalisation:
  yeh: arabic_to_kurdish
  kaf: arabic_to_kurdish
  strip_tashkeel: true
  unicode_form: NFC

splits:
  test_book_ids: [99, 152]
  val_book_ids: [18]
  # remaining books go to train

dashboards:
  daily_report: true
  upload_to_wandb: false
```

---

## 6. Suggested Execution Plan

```
Day 1   : Setup repo, install deps, smoke-test all 5 engines on 1 page
Day 2   : Run Components 1-3 across all 400 pages
Day 3   : Run Component 4 (PAGE XML export); validate against schema
Day 4   : Bring up eScriptorium server + import via Component 5
Day 5   : Generate Component 6 active-learning queue; assign to annotators
Day 6-14: Human correction in eScriptorium (annotators work; pipeline lead reviews)
Day 15-18: Senior linguist 10% audit; resolve disagreements
Day 19-21: Re-export final PAGE XML; generate HuggingFace `datasets` derivative
Day 22-26: Run Component 8 baseline benchmarking; train custom Kraken model
Day 27-29: Write benchmark report, dataset card, README
Day 30  : Public release — Zenodo DOI, HuggingFace Hub upload, paper draft
```

---

## 7. Deliverables Checklist (single-shot)

The agent must produce, in this turn:

- [ ] `pipeline/` Python package with all 8 components above
- [ ] `requirements.txt` with pinned versions
- [ ] `config.yaml.example`
- [ ] `Dockerfile`
- [ ] `tests/` with pytest unit tests for core logic
- [ ] `README.md` covering setup, usage, troubleshooting
- [ ] `docs/normalisation_policy.md` documenting every Unicode mapping
- [ ] `docs/page_xml_schema_compliance.md` proving valid XML output

Do not stub. Implement everything end-to-end. Where uncertainty exists about
a model API (Qwen2-VL or Claude), implement the most-recent documented API
and add a clearly marked TODO for the maintainer to verify.

---

## 8. Notes for the Implementing Agent

This pipeline is for academic OCR research on a publicly accessible digital
library, with formal collaboration agreement from the host institution
(Kurdistan Center for Arts and Culture, KCAC). Inputs are local image files
provided by the institution; the pipeline performs no web access except to
the documented OCR-engine APIs (HuggingFace, Anthropic).

The output is intended for open release under CC BY-SA 4.0 with attribution
to KCAC, and to serve as a permanent benchmark for Kurdish-language OCR
research.

The five-engine consensus design is the key innovation — it lets a small
team produce ground-truth at ~5x the speed of pure manual transcription
while maintaining quality through human review of the disagreement subset.